In [2]:
import pandas as pd
import numpy as np
import requests
import re
from io import StringIO
import xml.etree.ElementTree as ET
import json
import os

import tooleurostat as et
from tooleurostat import serial_db
import toolistat as ti
import tooloecd as to
from tooldb import clean_stat, update_json
from functools import reduce

In [3]:
import config

keys = config.KEYS
countries = config.COUNTRIES
start_year = config.START_YEAR
end_year = config.END_YEAR

In [4]:
def find_unique(df):
    for col in df.columns:
        if col != 'OBS_VALUE':
            print(col, df[col].unique())

## WIP Environment:

## International Trade

In [ ]:
# INTERNATIONAL TRADE FROM A DIFFERENT EUROPEAN DATA CENTER
# Filters : frequency.reporter.partner.product.flow.unit

dataset_code = "DS-059331"
filters = "A.IT.AE.04.1.VALUE_EUR"
start_year = 2020
end_year = 2022

if filters and not filters.startswith("/"):
    filters = f"/{filters}"

base_url = "https://ec.europa.eu/eurostat/api/comext/dissemination/sdmx/2.1/data"
url = f"{base_url}/{dataset_code}{filters}"

params = {
        "format": "SDMX-CSV",
    }

if start_year:
        params["startPeriod"] = start_year
if end_year:
    params["endPeriod"] = end_year

try:
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    df = pd.read_csv(StringIO(response.text), sep=',', low_memory=False, index_col=False)

except requests.RequestException as e:
    print(f"Request error: {e}")
except pd.errors.ParserError as e:
    print(f"CSV parsing error: {e}")

df.head()

,DATAFLOW,LAST UPDATE,freq,reporter,partner,product,flow,indicators,TIME_PERIOD,OBS_VALUE
0,ESTAT:DS-059331(1.0),15/09/25 11:00:00,A,IT,AE,4,1,VALUE_EUR,2020,116532
1,ESTAT:DS-059331(1.0),15/09/25 11:00:00,A,IT,AE,4,1,VALUE_EUR,2021,6876
2,ESTAT:DS-059331(1.0),15/09/25 11:00:00,A,IT,AE,4,1,VALUE_EUR,2022,45184


## Testing Eurostat Environment

## Pollution Analysis
- incgrp: A (rich) and B (poor) areas above and below 60% median income
- deg_urb: 1 (city), 2 (town), 3 (rural area)

In [20]:
# Pollution, grime or other environmental problems by degree of urbanisation
poll_df = et.get_eurostat_dataset(
    dataset_code="ilc_mddw05",
    start_year="2003",
    end_year="2023"
)
poll_df.head()

,freq,unit,deg_urb,incgrp,geo,TIME_PERIOD,OBS_VALUE
0,A,PC,DEG1,A_MD60,AL,2017-01-01,4.8
1,A,PC,DEG1,A_MD60,AL,2018-01-01,3.1
2,A,PC,DEG1,A_MD60,AL,2019-01-01,5.7
3,A,PC,DEG1,A_MD60,AL,2020-01-01,6.2
4,A,PC,DEG1,A_MD60,AT,2003-01-01,10.0


## Climate Watch

In [5]:
print("-"*10 + "SOURCE")
req_url = "https://www.climatewatchdata.org/api/v1/data/historical_emissions/data_sources"
data_list = requests.get(req_url).json()
for d in data_list["data"]:
    print(d["id"], d["name"])

print("-"*10 + "GAS")
gases_url = "https://www.climatewatchdata.org/api/v1/data/historical_emissions/gases"
gases_list = requests.get(gases_url).json()
for g in gases_list["data"]:
    print(g["id"], g["name"])

print("-"*10 + "SECTORS")
req_url = "https://www.climatewatchdata.org/api/v1/data/historical_emissions/sectors"
data_list = requests.get(req_url).json()
for d in data_list["data"]:
    print(d["id"], d["name"])


----------SOURCE
244 Climate Watch
245 PIK
246 UNFCCC_AI
247 UNFCCC_NAI
248 GCP
249 US
----------GAS
513 All GHG
514 CO2
515 CH4
516 N2O
517 F-Gas
518 KYOTOGHG
519 Aggregate GHGs
520 Aggregate F-gases
----------SECTORS
2916 Land-Use Change and Forestry
2917 Bunker Fuels
2918 Electricity/Heat
2919 Commercial
2920 Residential
2921 Industry
2922 Transportation
2923 Fugitive Emissions
2853 Total excluding LUCF
2854 Total including LUCF
2855 Energy
2856 Industrial Processes
2857 Agriculture
2858 Waste
2859 Land-Use Change and Forestry
2860 Bunker Fuels
2861 Electricity/Heat
2862 Manufacturing/Construction
2863 Transportation
2864 Building
2865 Other Fuel Combustion
2866 Fugitive Emissions
2867 Total including LULUCF
2868 Total excluding LULUCF
2869 Energy
2870 Industrial Processes and Product Use
2871 Agriculture
2872 Land-Use Change and Forestry
2873 Waste
2874 Fuel Combustion Activities
2875 Fugitive Emissions from Solid Fuels
2876 Fugitive Emissions from Oil and Gas
2877 Mineral Products

In [270]:
import requests
import pandas as pd

In [272]:
base_url = "https://www.climatewatchdata.org/api/v1/data/historical_emissions"
params = {
    "source_ids[]": 232,
    "gas_ids[]": 497,
    "regions[]": ["ITA", "DEU"]
}

records = []
url = base_url
while url:
    resp = requests.get(url, params=params)
    payload = resp.json()
    data = payload.get("data", [])
    for entry in data:
        for e in entry.get("emissions", []):
            records.append({
                "id": entry.get("id"),
                "iso_code3": entry.get("iso_code3"),
                "country": entry.get("country"),
                "data_source": entry.get("data_source"),
                "sector": entry.get("sector"),
                "gas": entry.get("gas"),
                "unit": entry.get("unit"),
                "year": e.get("year"),
                "value": e.get("value")
            })

    # Pagina successiva:
    link_header = resp.headers.get("Link", "")
    next_url = None
    for part in link_header.split(","):
        if 'rel="next"' in part:
            next_url = part.split(";")[0].strip()[1:-1]
    url = next_url
    params = {}  # A volte la next URL include i parametri

df = pd.DataFrame(records)
#df = clean_estat(df, keys=['iso_code3', 'year'], columns_to_pivot=['sector'], obs_value='value')
df.head()

""


In [275]:
payload

{'data': [],
 'meta': {'years': [],
  'header_years': [],
  'sorting': {'sort_col': 'iso_code3', 'sort_dir': 'ASC'},
  'columns': ['id', 'iso_code3', 'country', 'data_source', 'sector', 'gas']}}

In [ ]:
req_url = "https://data.emissionspathways.org/api/v1/data/emission_pathways/models"
data_list = requests.get(req_url).json()
for d in data_list["data"]:
    print(d["id"], d["full_name"]) # description

19 2050 Pathways Calculator
7 Annual Energy Outlook 2017
40 California Pathways Model
47 EnergyPATHWAYS
42 Energy Policy Simulator, Mexico
15 Energy Policy Simulator, United States
16 Global Calculator
3 Global Change Assessment Model
14 Global Multi-regional MARKAL
8 International Energy Outlook
46 MISO-Framework for Analysis of Climate-Energy-Technology Systems
22 National Reports submitted to the UN
41 Risky Business


In [ ]:
base_url = "https://data.emissionspathways.org/api/v1/data/emission_pathways"
params = {
    "model_ids[]": ,
    "scenario_ids[]": ,
    "category_ids[]": ,
    "indicator_ids[]": ,
    "location_ids[]": ,
}

records = []
url = base_url
while url:
    resp = requests.get(url, params=params)
    payload = resp.json()
    data = payload.get("data", [])
    for entry in data:
        for e in entry.get("emissions", []):
            records.append({
                "id": entry.get("id"),
                "iso_code3": entry.get("iso_code3"),
                "country": entry.get("country"),
                "data_source": entry.get("data_source"),
                "sector": entry.get("sector"),
                "gas": entry.get("gas"),
                "unit": entry.get("unit"),
                "year": e.get("year"),
                "value": e.get("value")
            })

    # Pagina successiva:
    link_header = resp.headers.get("Link", "")
    next_url = None
    for part in link_header.split(","):
        if 'rel="next"' in part:
            next_url = part.split(";")[0].strip()[1:-1]
    url = next_url
    params = {}  # A volte la next URL include i parametri

df = pd.DataFrame(records)
print(df["year"].unique())


## PIT

In [4]:
import tooloecd as to
import config
from tooldb import last_observed_per_group

keys = config.KEYS
countries = config.OECD_COUNTRIES

In [ ]:

# Personal income tax (PIT) - central government rates and thresholds
pit_tax_df = to.get_oecd_dataset(
    dataset_id= "OECD.CTP.TPS,DSD_TAX_PIT@DF_PIT_CENT,1.0",
    filters= f"{countries}.A.TH+MR_R.PIT..S1311.S.S_C0....",
    start_year=start_year,
    end_year=end_year
)

drop_cols = {'UNIT_MEASURE'}

pit_tax_df.drop(columns=drop_cols.intersection(pit_tax_df.columns), inplace=True)
pit_tax_df = pit_tax_df.rename(columns={'REF_AREA': 'geo'})
pit_tax_df = clean_stat(pit_tax_df, keys=['geo', 'TIME_PERIOD', 'LEVEL'], columns_to_pivot=['TRANSACTION', ])

pit_tax_df

TRANSACTION,geo,time_period,level,mr_r,th
0,AUS,2018,L1,0.0,18200.0
1,AUS,2018,L2,19.0,37000.0
2,AUS,2018,L3,32.5,87000.0
3,AUS,2018,L4,37.0,180000.0
4,AUS,2018,L5,45.0,None
...,...,...,...,...,...
859,USA,2023,L3,22.0,95375.0
860,USA,2023,L4,24.0,182100.0
861,USA,2023,L5,32.0,231250.0
862,USA,2023,L6,35.0,578125.0


In [ ]:
prod_df = to.get_oecd_dataset(
    dataset_id= "OECD.SDD.TPS,DSD_PDB@DF_PDB,2.0",
    filters= f"{countries}.A.SELF+SAL+GDPPOP+HRSAV+LCEMP+LCHRS+LCOST+LCOSTE+LCTOT+LCTOTE+ULCE+ULCH.A+BTE+BTNXL+B+C+D+E+F+GTNXL+G+H+I+J+K+L+M+N+O+P+Q+R+_T.XDC+XDC_H+PS+H_PS+PA+IX.V+_Z.N..",
    start_year="2000"
)

prod_df = prod_df.rename(columns={'REF_AREA': 'geo'})
prod_clean_df = clean_stat(prod_df, keys=keys, columns_to_pivot=['MEASURE', 'UNIT_MEASURE', 'ACTIVITY'])
prod_clean_df

## Testing OECD Environment

In [ ]:
# This database on asset-backed pensions is based on data collected through the OECD Global Pension Statistics exercise.
# Data come from various administrative sources, mainly: pension supervisory authorities, 
# financial market authorities, ministries of finance, or national statistical offices. 
# Data cover all asset-backed pension arrangements where assets are accumulated to back future benefit payments, 
# except reserves of public (pay-as-you-go) pension arrangements.

asset_pens_df = to.get_oecd_dataset(
    dataset_id= "OECD.DAF.CM,DSD_FP@DF_FPS,1.0",
    filters= f"{countries}.A.1290+1110+1122+1161+1210+1215+1240+1245+1260+1270+1121+1000.XDC.PER+OCC+_T._T.OTH+PIC+BR+PF+_T",
    start_year=2019,
    end_year=2024
)

drop_cols = {'UNIT_MEASURE'}

#asset_pens_df.drop(columns=drop_cols.intersection(asset_pens_df.columns), inplace=True)
#asset_pens_df = asset_pens_df.rename(columns={'REF_AREA': 'geo'})
#asset_pens_df = clean_stat(asset_pens_df, keys=['geo', 'TIME_PERIOD', 'LEVEL'], columns_to_pivot=['TRANSACTION', ])

asset_pens_df

,REF_AREA,FREQ,MEASURE,UNIT_MEASURE,PLAN_TYPE,DEFINITION_TYPE,VEHICLE_TYPE,TIME_PERIOD,OBS_VALUE
0,TUR,A,1121,XDC,PER,_T,_T,2020,3825.572
1,TUR,A,1110,XDC,PER,_T,_T,2020,5499.479
2,TUR,A,1290,XDC,PER,_T,_T,2020,16331.406
3,TUR,A,1000,XDC,PER,_T,_T,2020,169488.092
4,TUR,A,1215,XDC,PER,_T,_T,2020,108234.365
...,...,...,...,...,...,...,...,...,...
7119,PRT,A,1215,XDC,_T,_T,OTH,2023,1698.118
7120,PRT,A,1290,XDC,_T,_T,OTH,2023,-13.037
7121,PRT,A,1210,XDC,_T,_T,OTH,2023,100.602
7122,PRT,A,1000,XDC,_T,_T,OTH,2023,3816.160


In [ ]:
# Pensions at a Glance reviews and analyses the pension measures enacted or legislated in OECD countries
# for workers entering the labour market at age 22 in the specified year. 
# It provides an in-depth review of the first layer of protection of the elderly, 
# first-tier pensions across countries and provideds a comprehensive selection of 
# pension policy indicators for all OECD

pensions_df = to.get_oecd_dataset(
    dataset_id= "OECD.ELS.SPD,DSD_PAG@DF_PAG,1.0",
    filters= f"{countries}.A.GPRR200+GPRR50+NPRR50+NPRR200+GPW50+GPW200+NPW50+NPW200+CRPLF22+FRPLF22+GPRR100+ATRW+ATRAEP+ATRPPAE+NPRR100+GPW100+NPW100+FR+LE+ER+OAWAR+ELMEA+EYLME+EDIOP+PTOP+OCOP+CIOP+EIOP+AWGW+OAIP+PEP+PPEP....",
    start_year=1955,
    end_year=2022
)

drop_cols = {'UNIT_MEASURE', 'OPTIONALITY', 'FREQ'}

pensions_df

,REF_AREA,FREQ,MEASURE,UNIT_MEASURE,SEX,AGE,OPTIONALITY,TIME_PERIOD,OBS_VALUE
0,AUS,A,ELMEA,Y,M,_Z,_Z,1970,65.7
1,AUS,A,ELMEA,Y,M,_Z,_Z,1971,65.3
2,AUS,A,ELMEA,Y,M,_Z,_Z,1972,65.5
3,AUS,A,ELMEA,Y,M,_Z,_Z,1973,64.5
4,AUS,A,ELMEA,Y,M,_Z,_Z,1974,64.5
...,...,...,...,...,...,...,...,...,...
10703,ESP,A,OAWAR,PT_POP_Y20T64,_Z,Y_GE65,_Z,2022,33.4
10704,SWE,A,OAWAR,PT_POP_Y20T64,_Z,Y_GE65,_Z,2022,35.9
10705,TUR,A,OAWAR,PT_POP_Y20T64,_Z,Y_GE65,_Z,2022,14.2
10706,GBR,A,OAWAR,PT_POP_Y20T64,_Z,Y_GE65,_Z,2022,33.2


## Environment WIP SSP

In [106]:
dupes = df[df.duplicated(subset=['time_period', 'geo', 'model'], keep=False)]
print(dupes.sort_values(['time_period', 'geo', 'model']))

Empty DataFrame
Columns: [time_period, period, geo, model, avg-sup-temp-air]
Index: []


In [ ]:
import pandas as pd
from toolbi import default_connection
from tooldb import clean_stat

keys = ['MODEL', 'REGION', 'TIME_PERIOD']

ssp_emission_df = pd.read_csv('staticTables/environment/SSP_CMIP6_201811.csv')
ssp_emission_df = pd.melt(
    ssp_emission_df,
    id_vars=['MODEL', 'SCENARIO', 'REGION', 'VARIABLE', 'UNIT'],
    var_name="TIME_PERIOD",
    value_name="OBS_VALUE"
)

ssp_regions = ["R5.2ASIA", "R5.2LAM", "R5.2MAF", "R5.2OECD", "R5.2REF", "World"]
ssp_emission_df["TIME_PERIOD"] = pd.to_datetime(ssp_emission_df["TIME_PERIOD"])
ssp_emission_df['SCENARIO'] = ssp_emission_df['SCENARIO'].str.replace(' ', '-')
ssp_emission_df = ssp_emission_df[~ssp_emission_df["REGION"].isin(ssp_regions)]
ssp_emission_df['VARIABLE'] = (
    ssp_emission_df['VARIABLE']
        .str.replace('|', '_', regex=False)
        .str.replace(' ', '-', regex=False)
        .apply(lambda x: '_'.join(x.split('_')[:2] + ['total'] + x.split('_')[2:]) if x.count('_') != 2 else x)
        .apply(lambda x: x.replace('_', '-', 1))
)

ssp_emission_df = clean_stat(
                            ssp_emission_df, 
                            keys=keys, 
                            columns_to_pivot=['VARIABLE', 'SCENARIO'], 
                            obs_value='OBS_VALUE'
)
ssp_emission_df = ssp_emission_df.rename(columns={'region': 'geo'})

# Divide for each model and save
models = ssp_emission_df['model'].unique()

pattern = ["name", "type", "scenario", "model"]
descriptions = {
    'cmip6-emissions-cf4': 'Emissions (kt/yr) from CMIP6 model of Carbon Tetrafluorid: a long-lived fluorinated greenhouse gas (PFC) with strong radiative forcing.',
    'cmip6-emissions-co2': 'Emissions (Mt/yr) from CMIP6 model of Carbon Dioxid: the main anthropogenic greenhouse gas driving global warming.',
    'cmip6-emissions-bc': 'Emissions (Mt/yr) from CMIP6 model of Black Carbo: a short-lived climate forcer and aerosol component that absorbs sunlight (warming effect).',
    'cmip6-emissions-voc': 'Emissions (Mt/yr) from CMIP6 model of Volatile Organic Compounds (often NMVOC: ozone precursors that participate in tropospheric photochemistry.',
    'cmip6-emissions-ch4': 'Emissions (Mt/yr) from CMIP6 model of Methan: a potent greenhouse gas and ozone precursor with a shorter lifetime than CO2.',
    'cmip6-emissions-co': 'Emissions (Mt/yr) from CMIP6 model of Carbon Monoxid: an indirect greenhouse gas affecting atmospheric chemistry and methane lifetime.',
    'cmip6-emissions-nh3': 'Emissions (Mt/yr) from CMIP6 model of Ammoni: a precursor to secondary inorganic aerosols (e.g. ammonium sulfate/nitrate).',
    'cmip6-emissions-nox': 'Emissions (Mt/yr) from CMIP6 model of Nitrogen Oxides (NO + NO2: precursors to ozone and nitrate aerosols; affect atmospheric chemistry.',
    'cmip6-emissions-oc': 'Emissions (Mt/yr) from CMIP6 model of Organic Carbo: a component of particulate matter (PM); contributes to aerosol formation and cooling.',
    'cmip6-emissions-sulfur': 'Emissions (Mt/yr) from CMIP6 model of Sulfur Compounds (typically SO2: aerosol precursor that forms sulfate aerosols with a cooling effect.'
}

db_name_map = {
    "AIM/CGE": "ssp_emiss_aimcge",
    "GCAM4": "ssp_emiss_gcam4",
    "IMAGE": "ssp_emiss_image",
    "MESSAGE-GLOBIOM": "ssp_emiss_messglob",
    "REMIND-MAGPIE": "ssp_emiss_remmag",
}

for model in models:
    # filter & drop empty rows/cols
    df_model = (
        ssp_emission_df
        .query('model == @model')
        .dropna(axis=0, how='all')
        .dropna(axis=1, how='all')
        .copy()
    )

    if df_model.empty:
        # optionally log / warn and skip
        print(f"[info] no rows for model {model} — skipping.")
        continue

    # drop the 'model' column entirely as requested
    if 'model' in df_model.columns:
        df_model = df_model.drop(columns=['model'])

    # append model suffix to every column name (sanitize "/" to "-" etc.)
    suffix = "_" + model.replace("/", "-")
    df_model.columns = [
        f"{col}{suffix}" if col not in ["geo", "time_period"] else col for col in df_model.columns
    ]
    db_name = db_name_map.get(model)

    update_json(df_model, db_name, pattern, descriptions)

In [61]:
ssp_energy_df = pd.read_csv('staticTables/environment/SSP_IAM_V2_201811.csv')
ssp_energy_df = pd.melt(
    ssp_energy_df,
    id_vars=['MODEL', 'SCENARIO', 'REGION', 'VARIABLE', 'UNIT'],
    var_name="TIME_PERIOD",
    value_name="OBS_VALUE"
)
ssp_regions = []
ssp_vars = ['diagnostics', 'harmonized']
units = ssp_energy_df['UNIT'].unique()
ssp_energy_df["TIME_PERIOD"] = pd.to_datetime(ssp_energy_df["TIME_PERIOD"])
ssp_energy_df['VARIABLE'] = ssp_energy_df['VARIABLE'].str.replace('|', '_').str.replace(' ', '-')
ssp_energy_df = ssp_energy_df[~ssp_energy_df["REGION"].isin(ssp_regions)]
mask = ~ssp_energy_df['VARIABLE'].str.contains('|'.join(ssp_vars), case=False, na=False)
ssp_energy_df = ssp_energy_df[mask]

#ssp_energy_df = clean_stat(
#                            ssp_energy_df, 
#                            keys=keys, 
#                            columns_to_pivot=['VARIABLE'], 
#                            obs_value='OBS_VALUE'
#)
#ssp_energy_df = ssp_energy_df.rename(columns={'region': 'time_period'})

pattern = ['name', ]
descriptions = {
    'Agricultural-Demand': 'in million t DM/yr',
    'Capacity': 'in ',
    'Consumption': 'in billion US$2005/yr',
    'GDP_PPP': 'in ',
    'Emissions': 'in Mt BC/yr',
    'Final-Energy': 'in Mt of the chemical / yr, for N20 in kt',
    'Primary-Energy': 'in EJ/yr',
    'Secondary-Energy': 'in EJ/yr',
    'Land-Cover': 'in million ha (hectars)',
    'Population': 'in million people',
    'Price': 'in US$2005/t CO2',
    'Energy-Service': 'in bn tkm/yr',
    'Energy-Service': 'in bn pkm/yr',
}


In [62]:
ssp_energy_df

,MODEL,SCENARIO,REGION,VARIABLE,UNIT,TIME_PERIOD,OBS_VALUE
0,AIM/CGE,SSP1-19,R5.2ASIA,Agricultural-Demand_Crops,million t DM/yr,2005-01-01,1045.142000
1,AIM/CGE,SSP1-19,R5.2ASIA,Agricultural-Demand_Crops_Energy,million t DM/yr,2005-01-01,0.000000
2,AIM/CGE,SSP1-19,R5.2ASIA,Agricultural-Demand_Livestock,million t DM/yr,2005-01-01,85.998500
3,AIM/CGE,SSP1-19,R5.2ASIA,Agricultural-Production_Crops_Energy,million t DM/yr,2005-01-01,0.000000
4,AIM/CGE,SSP1-19,R5.2ASIA,Agricultural-Production_Crops_Non-Energy,million t DM/yr,2005-01-01,1329.199400
...,...,...,...,...,...,...,...
927878,WITCH-GLOBIOM,SSP5-Baseline,World,Secondary-Energy_Liquids,EJ/yr,2100-01-01,419.919395
927879,WITCH-GLOBIOM,SSP5-Baseline,World,Secondary-Energy_Liquids_Biomass,EJ/yr,2100-01-01,42.084453
927880,WITCH-GLOBIOM,SSP5-Baseline,World,Secondary-Energy_Liquids_Biomass_w/o-CCS,EJ/yr,2100-01-01,42.084453
927881,WITCH-GLOBIOM,SSP5-Baseline,World,Secondary-Energy_Liquids_Oil,EJ/yr,2100-01-01,153.616757
